# Practice 108 — Time Series Econometrics: ARIMA, VAR & Cointegration

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np

from src.datasets import (
    generate_ar_series,
    generate_arma_series,
    generate_cointegrated_pair,
    generate_random_walk,
    generate_spurious_pair,
    generate_trend_stationary,
    generate_var_system,
)
from src.plotting import (
    acf_pacf_plot,
    equilibrium_error_plot,
    impulse_response_plot,
    series_plot,
    spurious_regression_plot,
    tstat_distribution_plot,
)

## Phase 1 — Stationarity, spurious regression & unit roots

Two *independent* random walks, regressed on each other with an ordinary OLS fit.
Nothing connects them — yet watch the slope's t-statistic and the R^2.

In [ ]:
from src._01_stationarity_unit_roots import regress_levels

pair = generate_spurious_pair(n=300, seed=0)
spurious_fit = regress_levels(pair.y, pair.x)
print(f"slope={spurious_fit.beta[1]:+.4f}  t={spurious_fit.tstat[1]:+.2f}  R^2={spurious_fit.r2:.3f}")
fig = spurious_regression_plot(pair.x, pair.y, slope=spurious_fit.beta[1], intercept=spurious_fit.beta[0])
fig

### Exercise — `src/_01_stationarity_unit_roots.py :: durbin_watson`

Open `src/_01_stationarity_unit_roots.py`, read the `TODO(human)` block above the
function, implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_stationarity_unit_roots import durbin_watson

dw = durbin_watson(spurious_fit.resid)
print(f"Durbin-Watson: {dw:.4f}")
print(f"Granger-Newbold rule (R^2 > DW => spurious): {spurious_fit.r2 > dw}")

The single replication above could be a fluke. `simulate_spurious_regressions`
(fully scaffolded) reruns it 500 times with fresh independent random walks, so we
can see how often a nominal 5% test wrongly "rejects" the true null of no
relationship.

In [ ]:
from src._01_stationarity_unit_roots import rejection_rate, simulate_spurious_regressions

mc = simulate_spurious_regressions(n_sims=500, n=300, seed=0)
print(f"Rejection rate at nominal 5% (|t| > 1.96): {rejection_rate(mc['tstat']):.2%}")
fig = tstat_distribution_plot(mc["tstat"])
fig

### Exercise — `src/_01_stationarity_unit_roots.py :: adf_test`

Open `src/_01_stationarity_unit_roots.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below. We run it (and KPSS,
scaffolded) on two series that look similarly trending but have different unit-root
status — the whole point of testing rather than eyeballing.

In [ ]:
from src._01_stationarity_unit_roots import compare_with_statsmodels, kpss_summary

rw = generate_random_walk(300, seed=0)
trend_stat = generate_trend_stationary(300, seed=0)

print("[random walk]  (regression='c')")
compare_with_statsmodels(rw, lags=1, regression="c")
kstat, kp = kpss_summary(rw, regression="c")
print(f"KPSS: {kstat:.4f}  p={kp:.4f}   (null here is STATIONARITY)")

print("\n[trend-stationary]  (regression='ct')")
compare_with_statsmodels(trend_stat, lags=1, regression="ct")
kstat2, kp2 = kpss_summary(trend_stat, regression="ct")
print(f"KPSS: {kstat2:.4f}  p={kp2:.4f}")

## Phase 2 — ARIMA identification: ACF/PACF and conditional least squares

Box-Jenkins identification: an AR(p) process has a PACF that cuts off sharply after
lag p, while its ACF decays gradually. `identification_table` and the ACF/PACF plot
below are the same picture two ways.

In [ ]:
from src._02_arima_identification import identification_table

ar_data = generate_ar_series(n=400, seed=0)
print(f"True AR(2), phi = {ar_data.ar_true}")
print(identification_table(ar_data.series, nlags=8))
fig = acf_pacf_plot(ar_data.series, lags=20, title_prefix="AR(2)")
fig

### Exercise — `src/_02_arima_identification.py :: ar_conditional_least_squares`

Open `src/_02_arima_identification.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_arima_identification import ar_conditional_least_squares, compare_with_autoreg

ar_fit = ar_conditional_least_squares(ar_data.series, p=2)
print(f"const={ar_fit.const:+.4f}  phi={np.round(ar_fit.phi, 4)}")
compare_with_autoreg(ar_data.series, p=2)

An ARMA(2, 1) series is the mirror-image identification picture — both ACF and PACF
decay, because the MA term means no finite lag cuts either one off cleanly. Its MA
part can't be estimated by the conditional-least-squares regression above (its
regressors would be unobserved past errors), so we fit it with statsmodels' full
MLE and check the residuals with the Ljung-Box "diagnose" step of Box-Jenkins.

In [ ]:
from src._02_arima_identification import arima_reference, ljung_box

arma_data = generate_arma_series(n=400, seed=0)
print(f"True phi={arma_data.ar_true}  true theta={arma_data.ma_true}")
print(identification_table(arma_data.series, nlags=8))
fig = acf_pacf_plot(arma_data.series, lags=20, title_prefix="ARMA(2, 1)")
fig

In [ ]:
ref = arima_reference(arma_data.series, order=(2, 0, 1))
lb_stat, lb_pvalue = ljung_box(np.asarray(ref.resid), lags=10)
print(f"ARIMA(2,0,1) MLE params: {np.round(ref.params, 4)}")
print(f"Ljung-Box(10) on its residuals: Q={lb_stat:.3f}  p={lb_pvalue:.4f}  (large p = adequate)")

## Phase 3 — VAR systems, Granger causality & impulse responses

A bivariate VAR(1) where `x` Granger-causes `y` but not the reverse (built into the
true coefficient matrix, printed below).

In [ ]:
var_data = generate_var_system(n=300, seed=0)
print("True VAR(1) coefficient matrix (rows = equations for [y, x]):")
print(var_data.coef_matrix)
fig = series_plot({"y": var_data.y, "x": var_data.x}, title="Bivariate VAR(1) system")
fig

### Exercise — `src/_03_var_granger.py :: granger_causality_f_test`

Open `src/_03_var_granger.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below. Test both directions — the
asymmetry is the point.

In [ ]:
from src._03_var_granger import compare_with_statsmodels as compare_granger
from src._03_var_granger import fit_var, granger_causality_f_test, impulse_response

for label, (dep, indep) in (("x -> y", (var_data.y, var_data.x)), ("y -> x", (var_data.x, var_data.y))):
    res = granger_causality_f_test(dep, indep, lags=2)
    verdict = "REJECT no-causality" if res.p_value < 0.05 else "fail to reject"
    print(f"{label}:  F={res.f_stat:7.3f}  p={res.p_value:.4f}  -> {verdict}")

print("\nAgainst statsmodels (x -> y):")
compare_granger(var_data.y, var_data.x, lags=2)

The impulse response function traces out the dynamic path after a one-off shock —
where a VAR's economic interpretation actually lives. Fully scaffolded (`fit_var`,
`impulse_response`).

In [ ]:
results = fit_var(var_data.y, var_data.x, maxlags=2)
responses, lower, upper = impulse_response(results, periods=10)
fig = impulse_response_plot(responses, lower, upper, response_name="y", impulse_name="x")
fig

## Phase 4 — Cointegration: Engle-Granger two-step & the ECM

Two individually I(1) series that share one common stochastic trend — the
combination `y - beta*x` is stationary even though neither series is.

In [ ]:
coint_data = generate_cointegrated_pair(n=300, seed=0, beta=2.0)
print(f"True beta: {coint_data.beta_true}")
fig = series_plot({"y": coint_data.y, "x": coint_data.x}, title="Cointegrated pair (both I(1))")
fig

### Exercise — `src/_04_cointegration_ecm.py :: engle_granger_step1`

Open `src/_04_cointegration_ecm.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below. This is exactly Phase
1's spurious regression — legitimate here only because the residual turns out to
be stationary.

In [ ]:
from src._04_cointegration_ecm import compare_with_statsmodels as compare_eg
from src._04_cointegration_ecm import engle_granger_step1

eg = engle_granger_step1(coint_data.y, coint_data.x)
print(f"cointegrating beta_hat: {eg.beta:+.4f}")
compare_eg(coint_data.y, coint_data.x)
fig = equilibrium_error_plot(eg.equilibrium_error)
fig

### Exercise — `src/_04_cointegration_ecm.py :: error_correction_model`

Open `src/_04_cointegration_ecm.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below. `speed` must come out
negative — that's the mean-reversion signature.

In [ ]:
from src._04_cointegration_ecm import error_correction_model

ecm = error_correction_model(coint_data.y, coint_data.x, eg.equilibrium_error)
print(f"short-run effect of d(x): {ecm.short_run:+.4f}")
print(f"adjustment speed alpha:   {ecm.speed:+.4f}  (t={ecm.speed_tstat:+.2f})")
half_life = np.log(0.5) / np.log(1.0 + ecm.speed) if -2 < ecm.speed < 0 else float("nan")
print(f"implied half-life of a deviation: {half_life:.2f} periods")

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert 0.0 <= dw <= 4.0
assert abs(ar_fit.phi[0] - ar_data.ar_true[0]) < 0.15, "AR(1) coefficient should be close to the true 0.6"

res_xy = granger_causality_f_test(var_data.y, var_data.x, lags=2)
res_yx = granger_causality_f_test(var_data.x, var_data.y, lags=2)
assert res_xy.p_value < 0.05, "x should Granger-cause y"
assert res_yx.p_value > 0.05, "y should NOT Granger-cause x"

assert abs(eg.beta - coint_data.beta_true) < 0.3, "cointegrating beta should be close to the true 2.0"
assert ecm.speed < 0, "adjustment speed should be negative (mean-reverting)"
print("OK")